<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 6 · Low, Expected and High: an Honest 80% Range
### Every forecast hour gets three numbers. The leaderboard rewards a range that is narrow and honest.

*Solo lab · about 75 minutes · Colab · the Lab 5 data, then your own energy project on Google Cloud*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab6_Low_Expected_High.ipynb)

In Lab 5 your model made **one number** per hour. A grid operator needs more: Connecticut will **probably** use about
3,000 MW at 5 p.m. tomorrow, but how low or high **could** it go? Tonight every forecast hour gets three numbers:

| Column | Meaning |
|---|---|
| `predicted_low_mw` | the 10th percentile: the load should be **below** this only 1 hour in 10 |
| `predicted_load_mw` | the expected value (your Lab 5 number) |
| `predicted_high_mw` | the 90th percentile: the load should be **above** this only 1 hour in 10 |

Low to high is an **80% range**: about 8 of every 10 actual loads should land inside it.

| Part | Time | What you do |
|---|---|---|
| **0. Before you start** | 5 min | Sync your fork, so `forecast-24h` knows how to make a range |
| **1. Load** | 5 min | The Lab 5 data, features and split, plus a **calibration year** |
| **2. The expected value** | 5 min | Your Lab 5 champion (or a better one) |
| **3. What makes a good range** | 10 min | Coverage, width and the leaderboard score, worked out by hand |
| **4. Method A: quantile boosting** | 10 min | Two more models that aim at the 10th and 90th percentiles, with a loss curve |
| **5. Method B: offsets by hour** | 10 min | How far off was a model on a year it never saw, at each hour of the day? |
| **6. Compare and choose** | 10 min | Coverage, width and score on the test year; a real week with its range |
| **Your turn** | 5 min | Widen a range until it is honest (you write the code) |
| **7. Ship it** | 10 min | Upload the range with your model; run `forecast-24h` now |
| **8. Tomorrow** | 5 min | Your next 24 hours as a band, next to ISO New England's forecast |
| **9. Wrap up** | 5 min | Five short answers |

# Part 0 · Before you start (5 min)

`forecast-24h` learned to make a range after Lab 5, so your fork needs the update:

1. Open your fork of `energy-forecasting` on GitHub and click **Sync fork → Update branch**.
2. Click the **Actions** tab. A **Deploy forecast-24h** run starts by itself. Wait for the green check (about 3 minutes).

That's all. Until you ship a range in Part 7, `forecast-24h` keeps making your Lab 5 forecast with the low and high
columns empty. Nothing breaks.

# Part 1 · Load (5 min)

## 1a. Match production's library versions
The same scikit-learn version as `forecast-24h`. If Colab asks you to **restart the session**, click it, then run this
notebook again from the top.

In [ ]:
!pip install -q scikit-learn==1.6.1

import sklearn
print("scikit-learn", sklearn.__version__, "(production uses 1.6.1)")

## 1b. Your project and bucket
The **energy** project's ID and bucket, the same as Lab 5 Part 1b.

In [ ]:
PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"     # e.g. "energy-dww05002-f26"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"    # usually the same as your project ID

## 1c. The Lab 5 data and `make_features`, word for word
The same 11 features `forecast-24h` builds every hour (`cloud_function/forecast-24h/main.py` in your fork).

In [ ]:
import io
import json
import time
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])

WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]


def make_features(hours_utc, weather, load):
    """One row of model inputs per hour.
    hours_utc: the hours to describe (UTC).  weather: table indexed by UTC hour with the WEATHER columns.
    load: Connecticut load in MW, indexed by UTC hour (only hours that are already over)."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert("America/New_York")           # people use electricity on local time
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek                       # 0 = Monday ... 6 = Sunday
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)

    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values

    # Same hour yesterday. If that hour isn't over yet (only the last hour of a live forecast),
    # use the hour just before it.
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]


X_all = make_features(df.index, df, df["load_mw"])
y_all = df["load_mw"]
has_yesterday = X_all["load_yesterday_mw"].notna()
X_all = X_all[has_yesterday]
y_all = y_all[has_yesterday]
print(len(X_all), "hours with features, from", df["hour_local"].iloc[0], "to", df["hour_local"].iloc[-1])

## 1d. Three periods, not two
Lab 5 had **train** and **test**. A range needs one more period. To learn how far off a model is, we must watch it
forecast hours it **never saw**. So:

| Period | Years | Used for |
|---|---|---|
| **early** | 2023 to 2024 | fit a model whose misses we then watch |
| **calibration** | 2025 | watch that model miss; learn the range from its misses (Method B) |
| **test** | 2026 so far | the final exam, as in Lab 5: never used to choose anything |

Models that compete on the test year are trained on **2023 to 2025** (early + calibration), exactly like Lab 5.

In [ ]:
years = X_all.index.tz_convert("America/New_York").year
local_hour = X_all.index.tz_convert("America/New_York").hour
TEST_YEAR = years.max()
CAL_YEAR = TEST_YEAR - 1

early = years < CAL_YEAR
cal = years == CAL_YEAR
train = years < TEST_YEAR
test = years == TEST_YEAR

X_train = X_all[train]
y_train = y_all[train]
X_test = X_all[test]
y_test = y_all[test]
print(int(early.sum()), "early hours |", int(cal.sum()), "calibration hours (" + str(CAL_YEAR) + ") |",
      int(test.sum()), "test hours (" + str(TEST_YEAR) + " so far)")

# Part 2 · The expected value (5 min)
Your Lab 5 champion's settings. **Built a better model?** Change `new_model()` to return it (any scikit-learn
regressor on the 11 features). Every part below uses `new_model()`, so the whole lab follows your choice.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

SEED = 42
BEST_PARAMS = {"learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 31}    # <-- your Lab 5 champion's settings


def new_model():
    return HistGradientBoostingRegressor(random_state=SEED, **BEST_PARAMS)


started = time.time()
model = new_model()
history = model.fit(X_train, y_train)          # fit() returns the fitted model, kept as "history"
expected = model.predict(X_test)
test_mae = mean_absolute_error(y_test, expected)
yesterday_mae = mean_absolute_error(y_test, X_test["load_yesterday_mw"])
print("Expected value, test year: MAE", round(test_mae, 1), "MW (same hour yesterday:", round(yesterday_mae, 1), "MW)",
      "|", round(time.time() - started), "seconds")

**The loss curve** (same as Lab 5 Part 5d): fit on the earlier 80% of the training hours, watch the later 20%.

In [ ]:
cut = int(len(X_train) * 0.8)                      # earlier 80% = fit, later 20% = validation (never shuffled)
X_fit = X_train.iloc[:cut]
y_fit = y_train.iloc[:cut]
X_valid = X_train.iloc[cut:]
y_valid = y_train.iloc[cut:]

curve_model = new_model()
history = curve_model.fit(X_fit, y_fit)
fit_mae = []
valid_mae = []
for fit_pred in history.staged_predict(X_fit):
    fit_mae.append(mean_absolute_error(y_fit, fit_pred))
for valid_pred in history.staged_predict(X_valid):
    valid_mae.append(mean_absolute_error(y_valid, valid_pred))

trees = np.arange(1, len(fit_mae) + 1)
plt.figure(figsize=(8, 4))
plt.plot(trees, fit_mae, label="training hours")
plt.plot(trees, valid_mae, label="validation hours (later in time)")
plt.ylim(0, max(valid_mae[10], fit_mae[10]) * 1.2)
plt.xlabel("Number of trees")
plt.ylabel("MAE (MW)")
plt.title("Loss curve: the expected value, tree by tree")
plt.legend()
plt.show()

# Part 3 · What makes a good range (10 min)

Two things, and they pull against each other:
- **Honest (coverage):** about 80% of actual loads land inside the 80% range. Too few means **overconfident**.
- **Narrow (width):** a range from 0 to 10,000 MW is always right and useless.

The leaderboard puts both into **one score per forecast hour**, in MW, lower is better (the **weighted interval score**):

$$\text{score} = \frac{0.5 \times |\text{actual} - \text{expected}| \;+\; 0.1 \times (\text{width} + 10 \times \text{miss})}{1.5}$$

- `width` = high − low
- `miss` = how far the actual fell **outside** the range (0 when inside)
- the 10 is $2 / 0.2$: for an 80% range, every MW outside costs **10 times** as much as a MW of width

With no range (low = expected = high), the score is just the absolute error, the Lab 5 score. That's how every lab
ranks on one scale.

## A worked example: one hour, three forecasters
At 5 p.m. Connecticut used **3,100 MW**. All three expected 3,000 MW, so all three missed by 100 MW.

In [ ]:
def range_score(actual, low, expected, high):
    """The leaderboard's score for each hour (MW, lower is better). Works on single numbers or whole columns."""
    actual = np.asarray(actual, dtype=float)
    width = np.asarray(high, dtype=float) - np.asarray(low, dtype=float)
    miss = np.clip(np.asarray(low) - actual, 0, None) + np.clip(actual - np.asarray(high), 0, None)
    return (0.5 * np.abs(actual - np.asarray(expected)) + 0.1 * (width + 10 * miss)) / 1.5


example = pd.DataFrame({
    "forecaster": ["no range (Lab 5)", "too narrow", "honest", "too wide"],
    "low": [3000, 2980, 2850, 2000],
    "expected": [3000, 3000, 3000, 3000],
    "high": [3000, 3020, 3200, 4000],
})
example["actual"] = 3100
example["width"] = example["high"] - example["low"]
example["inside"] = (example["actual"] >= example["low"]) & (example["actual"] <= example["high"])
example["miss"] = np.clip(example["low"] - example["actual"], 0, None) + np.clip(example["actual"] - example["high"], 0, None)
example["score"] = range_score(example["actual"], example["low"], example["expected"], example["high"]).round(1)
example

**Read the table.** "Too wide" scores worse than no range at all: it paid for 2,000 MW of width to catch one hour.
"Too narrow" helps only a little: the actual still fell 80 MW outside, and each of those MW cost 10. "Honest" wins: it
caught the actual and paid for only 350 MW of width.

One hour can't tell you if a range is honest. Over **many** hours, about 80% should land inside. This helper scores a
whole period:

In [ ]:
def judge(name, actual, low, expected, high):
    low = np.minimum(low, expected)          # production does the same: the expected value stays inside its range
    high = np.maximum(high, expected)
    actual = np.asarray(actual, dtype=float)
    inside = (actual >= low) & (actual <= high)
    return {"range": name,
            "inside_pct": round(100 * float(np.mean(inside)), 1),
            "avg_width_mw": round(float(np.mean(high - low)), 1),
            "score_mw": round(float(np.mean(range_score(actual, low, expected, high))), 1)}


results = []
results.append(judge("no range (Lab 5)", y_test, expected, expected, expected))
pd.DataFrame(results)

# Part 4 · Method A: quantile boosting (10 min)
Lab 5's model aims at the **middle** of the load: it minimizes absolute error. Change one setting, the **loss**, and
gradient boosting aims at a percentile instead. With `loss="quantile", quantile=0.1` an hour where the actual is
**below** the prediction costs 9 times as much as an hour above, so the model learns to sit low: about 10% of loads
fall under it. Two models: one for the 10th percentile (low), one for the 90th (high). This loss is called the
**pinball loss**.

In [ ]:
def new_quantile_model(quantile):
    return HistGradientBoostingRegressor(loss="quantile", quantile=quantile, random_state=SEED, **BEST_PARAMS)


started = time.time()
low_model = new_quantile_model(0.1)
history_low = low_model.fit(X_train, y_train)
high_model = new_quantile_model(0.9)
history_high = high_model.fit(X_train, y_train)
low_a = low_model.predict(X_test)
high_a = high_model.predict(X_test)
results.append(judge("A: quantile boosting", y_test, low_a, expected, high_a))
print(round(time.time() - started), "seconds")
pd.DataFrame(results)

**The loss curve for the high model.** The training hours and the later validation hours, tree by tree, scored with
the pinball loss it trains on. Watch the gap between the two lines: that gap is the trouble.

In [ ]:
from sklearn.metrics import mean_pinball_loss

curve_high = new_quantile_model(0.9)
history = curve_high.fit(X_fit, y_fit)
fit_loss = []
valid_loss = []
fit_inside = []
valid_inside = []
for fit_pred in history.staged_predict(X_fit):
    fit_loss.append(mean_pinball_loss(y_fit, fit_pred, alpha=0.9))
    fit_inside.append(100 * float(np.mean(y_fit <= fit_pred)))
for valid_pred in history.staged_predict(X_valid):
    valid_loss.append(mean_pinball_loss(y_valid, valid_pred, alpha=0.9))
    valid_inside.append(100 * float(np.mean(y_valid <= valid_pred)))

trees = np.arange(1, len(fit_loss) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(trees, fit_loss, label="training hours")
ax[0].plot(trees, valid_loss, label="validation hours (later in time)")
ax[0].set_ylim(0, max(valid_loss[10], fit_loss[10]) * 1.2)
ax[0].set_xlabel("Number of trees")
ax[0].set_ylabel("Pinball loss (MW)")
ax[0].set_title("Loss curve: the 90th-percentile model")
ax[0].legend()
ax[1].plot(trees, fit_inside, label="training hours")
ax[1].plot(trees, valid_inside, label="validation hours (later in time)")
ax[1].axhline(90, color="gray", linestyle="--", label="goal: 90% below it")
ax[1].set_ylim(50, 100)
ax[1].set_xlabel("Number of trees")
ax[1].set_ylabel("% of actual loads below the high")
ax[1].set_title("Is the high really the 90th percentile?")
ax[1].legend()
plt.show()
print("After", len(trees), "trees:", round(fit_inside[-1], 1), "% of training hours below the high,",
      round(valid_inside[-1], 1), "% of validation hours")

**Why Method A is overconfident.** On the hours it trained on, the model is very good, so the 90th percentile of
**its own training misses** sits close to its predictions. On new hours it misses by more, and the range is too
narrow. That is the same overfitting you saw in Lab 5's loss curve, now showing up as a range that is too narrow.

# Part 5 · Method B: offsets by hour (10 min)
A simpler idea that works with **any** model: watch an honest forecaster miss, then use its misses as the range.
1. Fit `new_model()` on the **early** years only (2023 to 2024).
2. Let it forecast the **calibration** year (2025), which it never saw. Every hour gives one miss: actual − forecast.
3. For each hour of the day, take the 10th and 90th percentile of those misses. Those are the low and high
   **offsets**. For example, if at 1 p.m. the load came in more than 250 MW under the forecast 1 time in 10, then the
   1 p.m. low = expected − 250.

In [ ]:
started = time.time()
early_model = new_model()
history = early_model.fit(X_all[early], y_all[early])
misses = pd.Series(y_all[cal].values - early_model.predict(X_all[cal]))
low_offsets = misses.groupby(local_hour[cal]).quantile(0.1).round(1)
high_offsets = misses.groupby(local_hour[cal]).quantile(0.9).round(1)
print(round(time.time() - started), "seconds |", CAL_YEAR, "MAE of a model that never saw it:",
      round(float(misses.abs().mean()), 1), "MW")

plt.figure(figsize=(10, 4))
plt.bar(low_offsets.index, low_offsets.values, color="#1f77b4", label="low offset (10th percentile of the misses)")
plt.bar(high_offsets.index, high_offsets.values, color="#ff7f0e", label="high offset (90th percentile)")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(range(24))
plt.xlabel("Hour of the day (Connecticut time)")
plt.ylabel("MW from the expected value")
plt.title("How far off the forecast was in " + str(CAL_YEAR) + ", by hour")
plt.legend()
plt.show()

**Look at midday.** The offsets are widest around 10 a.m. to 3 p.m. That's rooftop solar: on a sunny day thousands of
roofs cover part of Connecticut's load, and our weather features (one airport, one cloud-cover number) only partly see
it. The model knows least about midday, and Method B's range says so.

In [ ]:
low_b = expected + low_offsets.values[local_hour[test]]
high_b = expected + high_offsets.values[local_hour[test]]
results.append(judge("B: offsets by hour", y_test, low_b, expected, high_b))
pd.DataFrame(results)

# Part 6 · Compare and choose (10 min)
**Read the table.** Both ranges cut the score a lot compared with no range. Check `inside_pct` against the goal of 80:
the method that learned from honest misses lands close to it; the method that learned from its own training hours does
not.

## 6a. Where each range is honest, hour by hour

In [ ]:
by_hour = pd.DataFrame({"hour": local_hour[test],
                        "A_inside": (y_test.values >= np.minimum(low_a, expected)) & (y_test.values <= np.maximum(high_a, expected)),
                        "B_inside": (y_test.values >= np.minimum(low_b, expected)) & (y_test.values <= np.maximum(high_b, expected)),
                        "A_width": np.maximum(high_a, expected) - np.minimum(low_a, expected),
                        "B_width": np.maximum(high_b, expected) - np.minimum(low_b, expected)})
hourly = by_hour.groupby("hour").mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hourly.index, 100 * hourly["A_inside"], marker="o", label="A: quantile boosting")
ax[0].plot(hourly.index, 100 * hourly["B_inside"], marker="o", label="B: offsets by hour")
ax[0].axhline(80, color="gray", linestyle="--", label="goal: 80%")
ax[0].set_ylim(40, 100)
ax[0].set_xlabel("Hour of the day (Connecticut time)")
ax[0].set_ylabel("% of test hours inside the range")
ax[0].set_title("Honest?")
ax[0].legend()
ax[1].plot(hourly.index, hourly["A_width"], marker="o", label="A: quantile boosting")
ax[1].plot(hourly.index, hourly["B_width"], marker="o", label="B: offsets by hour")
ax[1].set_xlabel("Hour of the day (Connecticut time)")
ax[1].set_ylabel("Average width (MW)")
ax[1].set_title("Narrow?")
ax[1].legend()
plt.show()

## 6b. The hottest week of the test year, with both ranges

In [ ]:
hottest = X_test["temp_f"].idxmax()
week = X_test.index[(X_test.index >= hottest - pd.Timedelta(days=3)) & (X_test.index <= hottest + pd.Timedelta(days=3))]
spot = X_test.index.isin(week)
week_local = week.tz_convert("America/New_York")

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].fill_between(week_local, np.minimum(low_a, expected)[spot], np.maximum(high_a, expected)[spot], alpha=0.3, label="A: 80% range")
ax[1].fill_between(week_local, np.minimum(low_b, expected)[spot], np.maximum(high_b, expected)[spot], alpha=0.3, color="#ff7f0e", label="B: 80% range")
for k in range(2):
    ax[k].plot(week_local, expected[spot], linewidth=1, label="expected")
    ax[k].plot(week_local, y_test[spot].values, color="black", linewidth=1, label="actual")
    ax[k].set_ylabel("Connecticut load (MW)")
    ax[k].legend(loc="upper left")
ax[0].set_title("The hottest week of " + str(TEST_YEAR) + ": method A (top) and method B (bottom)")
plt.show()

**Look at the peaks.** Method B's range is as wide on a mild afternoon as on the hottest one, because its offsets
depend only on the hour. On the hottest days the actual load climbs right out of it. An idea for the final project:
offsets by hour **and** by temperature, or by how high the expected load is.

## 6c. Your choice
The **lowest score** wins, the same rule as the leaderboard. Change `RANGE_METHOD` only if you have a reason you can
explain in Part 9.

In [ ]:
results = []
results.append(judge("no range (Lab 5)", y_test, expected, expected, expected))
results.append(judge("A: quantile boosting", y_test, low_a, expected, high_a))
results.append(judge("B: offsets by hour", y_test, low_b, expected, high_b))
table = pd.DataFrame(results)
RANGE_METHOD = "quantile_models"
chosen = table.iloc[1]
if table["score_mw"].iloc[2] < table["score_mw"].iloc[1]:
    RANGE_METHOD = "offsets_by_hour"
    chosen = table.iloc[2]
# RANGE_METHOD = "quantile_models"      # <-- uncomment to override: "quantile_models" (A) or "offsets_by_hour" (B)
if RANGE_METHOD == "quantile_models":
    chosen = table.iloc[1]
else:
    chosen = table.iloc[2]
print("Shipping:", chosen["range"], "->", RANGE_METHOD)
table

## Your turn: make a range honest (5 min, you write the code)
`range_widen` stretches a range around the expected value: 1.0 = as is, 1.3 = 30% wider on both sides. Production
reads it from your model card, so you can fix a range **without retraining**.

Find the smallest `widen` (try 1.0, 1.05, 1.1, ... up to 2.0) that puts at least 80% of the **calibration** year inside
Method A's range. Use the calibration year, never the test year: choosing on the test year is cheating on the exam.
Then score Method A with that `widen` on the test year with `judge`. Did the score get better or worse? Why?

Hints: you need Method A models fit on the **early** years and their predictions for the calibration year, and
`low = expected - widen * (expected - low)`, `high = expected + widen * (high - expected)`.

In [ ]:
# YOUR CODE HERE: find the widen for Method A on the calibration year, then judge it on the test year

# Part 7 · Ship it (10 min)

## 7a. The final models and the model card
Like Lab 5: the expected-value model is refit on **all** the data (2023 to Oct 15, 2026). The card says how
`forecast-24h` should make the range:

| `range_method` | What gets uploaded |
|---|---|
| `"offsets_by_hour"` (B) | only `model.joblib`; the 24 low and 24 high offsets travel inside `model_card.json` |
| `"quantile_models"` (A) | `model.joblib` plus `model_low.joblib` and `model_high.joblib`, refit on all the data |

In [ ]:
RANGE_WIDEN = 1.0          # <-- 1.0 = as tested; your turn may suggest more

final_model = new_model()
history = final_model.fit(X_all, y_all)
joblib.dump(final_model, "/content/model.joblib", compress=3)
files = ["model.joblib"]
if RANGE_METHOD == "quantile_models":
    final_low = new_quantile_model(0.1)
    history_low = final_low.fit(X_all, y_all)
    final_high = new_quantile_model(0.9)
    history_high = final_high.fit(X_all, y_all)
    joblib.dump(final_low, "/content/model_low.joblib", compress=3)
    joblib.dump(final_high, "/content/model_high.joblib", compress=3)
    files.append("model_low.joblib")
    files.append("model_high.joblib")

best_params = {}
for key in BEST_PARAMS:
    best_params[key] = str(BEST_PARAMS[key])
card = {
    "model_type": type(final_model).__name__,
    "best_params": best_params,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "test_mae_mw": round(float(test_mae), 1),
    "baseline_same_hour_yesterday_mae_mw": round(float(yesterday_mae), 1),
    "range_method": RANGE_METHOD,
    "range_quantiles": [0.1, 0.9],
    "range_widen": RANGE_WIDEN,
    "test_range_inside_pct": float(chosen["inside_pct"]),
    "test_range_width_mw": float(chosen["avg_width_mw"]),
    "test_score_mw": float(chosen["score_mw"]),
    "seed": SEED,
}
if RANGE_METHOD == "offsets_by_hour":
    card["range_offsets_mw"] = {"low": low_offsets.tolist(), "high": high_offsets.tolist()}
with open("/content/model_card.json", "w") as f:
    json.dump(card, f, indent=2)
files.append("model_card.json")
print("Saved in /content:", files)
card

## 7b. Sign in and check (3 checks)
The same checks as Lab 5 Part 7b. When the **"Allow access"** pop-up opens, pick the **Gmail you used for Google
Cloud**. No energy project? Set `UPLOAD = False`: Part 8 runs in demo mode.

In [ ]:
UPLOAD = True

if UPLOAD and ("YOUR-ENERGY-PROJECT-ID" in PROJECT_ID or "YOUR-ENERGY-PROJECT-ID" in BUCKET_NAME):
    print("Type your PROJECT_ID and BUCKET_NAME in Part 1b, re-run that cell, then re-run this one.")
    print("Staying in demo mode until you do.")
    UPLOAD = False

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()

    account_lines = !gcloud config get-value account
    signed_in_as = "unknown"
    if len(account_lines) > 0:
        signed_in_as = account_lines[-1].strip()
    print("1. Signed in as:", signed_in_as)

    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    try:
        bucket_ok = bucket.exists()
    except Exception as err:
        bucket_ok = False
        print("   Google says:", str(err)[:200])
    if not bucket_ok:
        raise RuntimeError("The account " + signed_in_as + " cannot see the bucket " + BUCKET_NAME + ". "
                           "Check BUCKET_NAME in Part 1b, or Runtime > Disconnect and delete runtime, then run again "
                           "and pick your Google Cloud Gmail in the pop-up.")
    print("2. Bucket found:", BUCKET_NAME)

    state_lines = !gcloud functions describe forecast-24h --gen2 --region=us-central1 --project={PROJECT_ID} --format="value(state)"
    state = "NOT FOUND"
    if len(state_lines) > 0 and state_lines[-1].strip() in ["ACTIVE", "DEPLOYING", "FAILED"]:
        state = state_lines[-1].strip()
    print("3. forecast-24h:", state)
    if state != "ACTIVE":
        print("   Go to Part 0: Sync fork, wait for 'Deploy forecast-24h' to go green, then re-run this cell.")

## 7c. Upload, then run `forecast-24h` now
The live files are replaced; `energy/models/history/` keeps a copy of each, named by time. Then the same wait-and-check
as Lab 5 Part 7d, plus one more check: are the **low and high columns** filled?

In [ ]:
from zoneinfo import ZoneInfo

if UPLOAD:
    stamp = card["trained_at"].replace(":", "-")
    for name in files:
        bucket.blob("energy/models/" + name).upload_from_filename("/content/" + name)
        bucket.blob("energy/models/history/" + name.replace(".", "_" + stamp + ".", 1)).upload_from_filename("/content/" + name)
        print("Live:", "gs://" + BUCKET_NAME + "/energy/models/" + name)

    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(16):                 # every 15 seconds, for up to 4 minutes
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if not ready:
        print("No new forecast after 4 minutes. Run the log cell below and paste the log into Claude.")
    else:
        mine = pd.read_csv(io.BytesIO(latest_blob.download_as_bytes()))
        print("Done at", latest_blob.updated.astimezone(ZoneInfo("America/New_York")).strftime("%I:%M %p"), "(Eastern)")
        if "predicted_low_mw" not in mine.columns:
            print("No low/high columns: your forecast-24h is the Lab 5 version. Do Part 0 (Sync fork), then re-run this cell.")
        elif mine["predicted_low_mw"].isna().all():
            print("Low/high are empty. range_method in your forecast:", mine["range_method"].iloc[0], "- run the log cell below.")
        else:
            print("Range is live:", mine["range_method"].iloc[0], "| average width",
                  round(float((mine["predicted_high_mw"] - mine["predicted_low_mw"]).mean())), "MW")
else:
    print("UPLOAD is False: nothing uploaded. Part 8 will run in demo mode.")

**Optional, only for troubleshooting.** The newest log lines of `forecast-24h`. Look for `"range_method"` and
`"range_error"` in the newest line.

In [ ]:
if UPLOAD:
    !gcloud functions logs read forecast-24h --region=us-central1 --gen2 --limit=10 --project={PROJECT_ID}
else:
    print("UPLOAD is False: no log to read.")

# Part 8 · Tomorrow (5 min)
Your next 24 hours as a band, with the last two days of actual load and ISO New England's own forecast. No upload?
Demo mode shows the last day of the test period with Method B.

In [ ]:
# PLUMBING (run me): load your forecast and the live data, or build a demo from the test year


def read_bucket_csv(key):
    blob = bucket.blob(key)
    if not blob.exists():
        return None
    return pd.read_csv(io.BytesIO(blob.download_as_bytes()))


LIVE = False
isone = None
if UPLOAD and latest_blob.exists():
    mine = read_bucket_csv("energy/forecasts/latest.csv")
    mine["target_hour_utc"] = pd.to_datetime(mine["target_hour_utc"], utc=True)
    mine = mine.set_index("target_hour_utc")
    forecast = mine["predicted_load_mw"]
    band_low = forecast
    band_high = forecast
    if "predicted_low_mw" in mine.columns and mine["predicted_low_mw"].notna().any():
        band_low = mine["predicted_low_mw"]
        band_high = mine["predicted_high_mw"]
    actual_table = read_bucket_csv("energy/datasets/ct_load_5min_hourly_master.csv")
    actual_table["hour_utc"] = pd.to_datetime(actual_table["hour_utc"], utc=True)
    actual = actual_table.set_index("hour_utc")["load_mw_5min_avg"].sort_index().tail(48)
    isone_table = read_bucket_csv("energy/datasets/isone_ct_forecast_master.csv")
    if isone_table is not None:
        isone_table["target_hour_utc"] = pd.to_datetime(isone_table["target_hour_utc"], utc=True)
        isone_table = isone_table.sort_values(["target_hour_utc", "issued_at_utc"])
        isone = isone_table.drop_duplicates("target_hour_utc", keep="last").set_index("target_hour_utc")["isone_forecast_mw"]
        isone = isone.reindex(forecast.index)
    LIVE = True
    print("LIVE: your forecast, issued", mine["issued_at_utc"].iloc[0], "(UTC)")
else:
    last_day = X_test.index[-24:]
    forecast = pd.Series(expected[-24:], index=last_day)
    band_low = pd.Series(np.minimum(low_b[-24:], expected[-24:]), index=last_day)
    band_high = pd.Series(np.maximum(high_b[-24:], expected[-24:]), index=last_day)
    actual = y_test.loc[X_test.index[-72:-24]]
    print("DEMO: the last day of the test period with Method B (upload in Part 7 for the real thing)")

In [ ]:
hours_local = forecast.index.tz_convert("America/New_York")
plt.figure(figsize=(11, 4))
plt.plot(actual.index.tz_convert("America/New_York"), actual.values, color="black", label="actual load (last 48 hours)")
plt.fill_between(hours_local, band_low.values, band_high.values, alpha=0.3, label="YOUR 80% range (low to high)")
plt.plot(hours_local, forecast.values, marker="o", label="YOUR expected value")
if isone is not None and isone.notna().any():
    plt.plot(isone.index.tz_convert("America/New_York"), isone.values, linestyle="--", label="ISO New England's forecast")
plt.ylabel("Connecticut load (MW)")
plt.title("Your next 24 hours: low, expected and high")
plt.legend(loc="upper left")
plt.show()
peak = forecast.idxmax()
print("Peak:", round(forecast.max()), "MW on", peak.tz_convert("America/New_York").strftime("%a %I %p"),
      "| 80% range", round(band_low[peak]), "to", round(band_high[peak]), "MW")

# Part 9 · Wrap up (5 min)

## Watch it in production
Your range now goes on the class leaderboard every hour, with two new columns: **range coverage** (the share of
actual loads that landed inside your range) and **range width**. Check it over the next few days. Expect coverage
**below** your test-year number: Parts 4 to 6 used the weather that actually happened, but production only has the
weather **forecast**, which is wrong too. If coverage stays well under 80%, widen your range: change `"range_widen"` in
`model_card.json` (for example to 1.3) and upload only the card. No retraining, and `forecast-24h` uses it at its next
run.

## Five short answers (1 or 2 sentences each)
1. Part 3: why did "too wide" score worse than no range at all, and why did "honest" win?
2. Part 4: on the loss curve, what share of training hours fell below the high model, and what share of validation
   hours? What does the gap tell you?
3. Part 6: which method had the lowest test-year score, how close was its coverage to 80%, and at which hours of the
   day was the range widest? Why there?
4. Your turn: what `widen` made Method A honest on the calibration year, and did it help its test-year score?
5. Part 8: what is your range at tomorrow's peak hour? Is ISO New England's forecast inside it?

*Your answers:*

---

**Data credits:** weather from the Iowa State Mesonet (mesonet.agron.iastate.edu) and the National Weather Service
(forecast.weather.gov); load and forecasts from ISO New England (iso-ne.com, ISO Express web services).